<a href="https://colab.research.google.com/github/shevchenkoiv/analytics/blob/main/GitHub_6_base_stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.kaggle.com/datasets/ashyou09/indian-patient-disease-and-treatment-dataset?select=indian_diseases_dataset.csv
### Набор данных о заболеваниях и лечении индийских пациентов

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

from itertools import combinations
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/indian_diseases_dataset.csv')
df.head()

,patient_id,age,age_group,gender,state,city,region,urban_rural,disease_name,disease_category,...,insurance_status,treatment_type,hospital_type,days_hospitalized,treatment_cost_inr,outcome,death_flag,cause_of_death,recovery_days,follow_up_required
0,IND-00000001,70,70-79,Female,Bihar,Patna,East,Semi-Urban,Diarrhea,Waterborne,...,Employer,Medication,Government,21,102.0,Recovering,0,NaN,42.0,Yes
1,IND-00000002,27,20-29,Other,Gujarat,Surat,West,Urban,Chikungunya,Vector-Borne,...,Uninsured,Medication,Government,0,1151.0,Recovered,0,NaN,18.0,Yes
2,IND-00000003,66,60-69,Other,Jharkhand,Bokaro,East,Rural,Hypertension,Non-Communicable,...,Private,Medication,Government,2,89.0,Chronic Management,0,NaN,NaN,Yes
3,IND-00000004,27,20-29,Female,Karnataka,Hubli,South,Rural,Tuberculosis,Infectious,...,Private,Surgery,Government,9,1269.0,Recovering,0,NaN,69.0,Yes
4,IND-00000005,40,40-49,Male,Karnataka,Belagavi,South,Rural,Chronic Kidney Disease,Non-Communicable,...,Ayushman Bharat,Palliative,Private,31,670576.0,Deceased,1,Complications from Chronic Kidney Disease,NaN,No


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   patient_id          20000 non-null  object 
 1   age                 20000 non-null  int64  
 2   age_group           20000 non-null  object 
 3   gender              20000 non-null  object 
 4   state               20000 non-null  object 
 5   city                20000 non-null  object 
 6   region              20000 non-null  object 
 7   urban_rural         20000 non-null  object 
 8   disease_name        20000 non-null  object 
 9   disease_category    20000 non-null  object 
 10  severity            20000 non-null  object 
 11  diagnosis_date      20000 non-null  object 
 12  year                20000 non-null  int64  
 13  month               20000 non-null  object 
 14  season              20000 non-null  object 
 15  symptoms            20000 non-null  object 
 16  como

In [5]:
df.describe()

,age,year,bmi,days_hospitalized,treatment_cost_inr,death_flag,recovery_days
count,20000.000000,20000.000000,20000.000000,20000.000000,2.000000e+04,20000.000000,13693.000000
mean,44.061050,2021.498000,23.485495,9.771150,1.079405e+05,0.091100,33.205141
std,25.165393,2.294179,4.666045,12.078476,4.940645e+05,0.287758,26.427051
min,0.000000,2018.000000,14.000000,0.000000,1.200000e+01,0.000000,3.000000
25%,23.000000,2019.000000,19.700000,2.000000,1.421000e+03,0.000000,14.000000
50%,44.000000,2022.000000,23.200000,6.000000,6.604500e+03,0.000000,24.000000
75%,63.000000,2023.000000,27.000000,14.000000,3.933375e+04,0.000000,47.000000
max,95.000000,2025.000000,40.000000,60.000000,1.537582e+07,1.000000,100.000000


### 1. Feature Engineering

In [6]:
df.head()

,patient_id,age,age_group,gender,state,city,region,urban_rural,disease_name,disease_category,...,insurance_status,treatment_type,hospital_type,days_hospitalized,treatment_cost_inr,outcome,death_flag,cause_of_death,recovery_days,follow_up_required
0,IND-00000001,70,70-79,Female,Bihar,Patna,East,Semi-Urban,Diarrhea,Waterborne,...,Employer,Medication,Government,21,102.0,Recovering,0,NaN,42.0,Yes
1,IND-00000002,27,20-29,Other,Gujarat,Surat,West,Urban,Chikungunya,Vector-Borne,...,Uninsured,Medication,Government,0,1151.0,Recovered,0,NaN,18.0,Yes
2,IND-00000003,66,60-69,Other,Jharkhand,Bokaro,East,Rural,Hypertension,Non-Communicable,...,Private,Medication,Government,2,89.0,Chronic Management,0,NaN,NaN,Yes
3,IND-00000004,27,20-29,Female,Karnataka,Hubli,South,Rural,Tuberculosis,Infectious,...,Private,Surgery,Government,9,1269.0,Recovering,0,NaN,69.0,Yes
4,IND-00000005,40,40-49,Male,Karnataka,Belagavi,South,Rural,Chronic Kidney Disease,Non-Communicable,...,Ayushman Bharat,Palliative,Private,31,670576.0,Deceased,1,Complications from Chronic Kidney Disease,NaN,No


In [7]:
df.columns

Index(['patient_id', 'age', 'age_group', 'gender', 'state', 'city', 'region',
       'urban_rural', 'disease_name', 'disease_category', 'severity',
       'diagnosis_date', 'year', 'month', 'season', 'symptoms', 'comorbidity',
       'smoking_status', 'alcohol_use', 'bmi', 'blood_group', 'income_level',
       'insurance_status', 'treatment_type', 'hospital_type',
       'days_hospitalized', 'treatment_cost_inr', 'outcome', 'death_flag',
       'cause_of_death', 'recovery_days', 'follow_up_required'],
      dtype='object')

In [8]:
for i in df.columns:
    if df[i].isnull().sum() != 0:
        print(df[i].value_counts(dropna=False))
        print(df[i].isnull().sum())
        print()

comorbidity
NaN               6599
Kidney Disease    2314
Diabetes          2242
Obesity           2236
Multiple          2219
Heart Disease     2217
Hypertension      2173
Name: count, dtype: int64
6599

alcohol_use
Occasional    6697
NaN           6627
Heavy         3343
Regular       3333
Name: count, dtype: int64
6627

cause_of_death
NaN                                          18178
Complications from Stroke                      193
Complications from Heart Disease               149
Complications from Oral Cancer                 148
Complications from Chronic Kidney Disease      141
Complications from Lung Cancer                 138
Complications from Breast Cancer               123
Complications from Hypertension                112
Complications from Diabetes (Type 2)            90
Complications from COPD                         89
Complications from Pneumonia                    69
Complications from Asthma                       60
Complications from Swine Flu (H1N1)             

In [9]:
# Добавление столбцов с заменой пропущенных значений.
# Пропуск в comorbidity - это отсутствие зафиксированных побочных заболеваний.
def flag_has_comorbidity(comorbidity):
    if pd.isna(comorbidity):
        return 0
    else:
        return 1
df['has_comorbidity'] = df['comorbidity'].apply(flag_has_comorbidity)
df['has_comorbidity']

,has_comorbidity
0,0
1,1
2,1
3,0
4,1
...,...
19995,1
19996,1
19997,1
19998,1


In [10]:
def flag_has_recovery_days(recovery):
    if pd.isna(recovery):
        return 0
    else:
        return 1
df['has_recovery_days'] = df['recovery_days'].apply(flag_has_recovery_days)
df['has_recovery_days']

,has_recovery_days
0,1
1,1
2,0
3,1
4,0
...,...
19995,0
19996,1
19997,0
19998,0


In [11]:
def alcohol_use_value(alcohol):
    if pd.isna(alcohol):
        return "Unknown"
    else:
        return alcohol

df['alcohol_use_clean'] = df['alcohol_use'].apply(alcohol_use_value)
df['alcohol_use_clean']

,alcohol_use_clean
0,Unknown
1,Regular
2,Unknown
3,Occasional
4,Regular
...,...
19995,Occasional
19996,Occasional
19997,Occasional
19998,Heavy


In [12]:
!pip install forex_python
import forex_python

In [13]:
# Конвертация рупи в доллары
from forex_python.converter import CurrencyRates
c = CurrencyRates()
rate_1 = c.get_rate('INR', 'USD')
def int_to_usd(value_inr, rate):
    converted_amount = value_inr * rate
    return round(converted_amount, 2)
df['treatment_cost_usd'] = df['treatment_cost_inr'].apply(lambda x: int_to_usd(value_inr=x, rate=rate_1))
df[['treatment_cost_inr', 'treatment_cost_usd']]

,treatment_cost_inr,treatment_cost_usd
0,102.0,1.08
1,1151.0,12.18
2,89.0,0.94
3,1269.0,13.43
4,670576.0,7096.71
...,...,...
19995,44863.0,474.79
19996,1590.0,16.83
19997,11361.0,120.23
19998,3307.0,35.00


In [14]:
# Индекс массы тела по группам
def bmi_group(bmi):
    if bmi < 15:
        return '<15'
    elif 15 <= bmi < 20:
        return '15-19'
    elif 20 <= bmi < 25:
        return '20-24'
    elif 25 <= bmi < 30:
        return '25-29'
    elif 30 <= bmi < 35:
        return '30-34'
    elif 35 <= bmi < 40:
        return '35-39'
    else:
        return '>=40'

df['bmi_group'] = df['bmi'].apply(bmi_group)
df['bmi_group']

,bmi_group
0,20-24
1,15-19
2,25-29
3,15-19
4,25-29
...,...
19995,15-19
19996,20-24
19997,15-19
19998,30-34


In [15]:
cv = []
for i in df.columns:
    if i not in ['comorbidity', 'alcohol_use', 'cause_of_death']:
        print(i, df[i].min(), df[i].max())
    else:
        cv.append(i)
print()
for i in cv:
    print(df[i].unique())

patient_id IND-00000001 IND-00020000
age 0 95
age_group 0-9 80+
gender Female Other
state Andaman and Nicobar Islands West Bengal
city Agartala Warangal
region Central West
urban_rural Rural Urban
disease_name Anxiety Disorder Typhoid
disease_category Genetic Waterborne
severity Critical Severe
diagnosis_date 2018-01-01 2025-12-28
year 2018 2025
month April September
season Monsoon Winter
symptoms Abdominal pain, Cramps Wheezing, Sore throat, Shortness of breath, Loss of smell/taste, Fatigue
smoking_status Current Never
bmi 14.0 40.0
blood_group A+ O-
income_level Below Poverty Line Upper-Middle
insurance_status Ayushman Bharat Uninsured
treatment_type Home Remedies Therapy
hospital_type AIIMS / Apex Institution Private
days_hospitalized 0 60
treatment_cost_inr 12.0 15375816.0
outcome Chronic Management Recovering
death_flag 0 1
recovery_days 3.0 100.0
follow_up_required No Yes
has_comorbidity 0 1
has_recovery_days 0 1
alcohol_use_clean Heavy Unknown
treatment_cost_usd 0.13 162722.26
b

In [16]:
df.columns

Index(['patient_id', 'age', 'age_group', 'gender', 'state', 'city', 'region',
       'urban_rural', 'disease_name', 'disease_category', 'severity',
       'diagnosis_date', 'year', 'month', 'season', 'symptoms', 'comorbidity',
       'smoking_status', 'alcohol_use', 'bmi', 'blood_group', 'income_level',
       'insurance_status', 'treatment_type', 'hospital_type',
       'days_hospitalized', 'treatment_cost_inr', 'outcome', 'death_flag',
       'cause_of_death', 'recovery_days', 'follow_up_required',
       'has_comorbidity', 'has_recovery_days', 'alcohol_use_clean',
       'treatment_cost_usd', 'bmi_group'],
      dtype='object')

In [17]:
# df[['has_comorbidity', 'has_recovery_days', 'treatment_cost_usd']].describe()
# df['bmi_group'].describe(include='all')

Вывод:
1. Были созданы дополнительные признаки для дальнейшего анализа: has_comorbidity, has_recovery_days, alcohol_use_clean, treatment_cost_usd и bmi_group.
2. Пропуски в alcohol_use выделены в отдельную категорию Unknown.
3. Пропуски в recovery_days имеют логическую природу и не должны заменяться средним, так как этот показатель отсутствует у хронических и умерших пациентов.

### 2. EDA

In [18]:
# Описание пациентов и заболеваний
# общая смертность
# доля восстановившихся
# структура outcome
df["outcome"].value_counts(normalize=True).mul(100).round(2)  # доля каждой категории в столбце исходов(outcome) в процентах
mortality_rate = float((df['death_flag'].mean()).round(4))
mortality_rate_pct = float((df['death_flag'].mean() * 100).round(2))
recovery_rate = float(pd.to_numeric(df['outcome'] == 'Recovered', errors='coerce').mean().round(4))
recovery_rate_pct = float((pd.to_numeric(df['outcome'] == 'Recovered', errors='coerce').mean() * 100).round(2))
mortality_rate, mortality_rate_pct, recovery_rate, recovery_rate_pct

(0.0911, 9.11, 0.422, 42.2)

Вывод:
1. Общая смертность составляет - 9.11%.
2. Доля полностью восстановившихся пациентов - 42.2%.

### 3. Продуктовые метрики

In [19]:
# Продуктовые метрики:
# Средняя стоимость лечение.
# Стоимость одного выздоровевшего пациента.
# Средняя продолжительность лечения.

In [20]:
avg_cost_usd = float((df['treatment_cost_usd'].mean()).round(2))
median_cost_usd = round(df['treatment_cost_usd'].median(), 2)
avg_hospital_days = float((df['days_hospitalized'].mean()).round(2))
# Средняя стоимость лечения среди выздоровевших пациентов
cost_per_recovered_patient = float(df[df['outcome'] == 'Recovered']['treatment_cost_usd'].mean().round(2))
business_metrics = pd.DataFrame({
    "metric": [
        "Средняя стоимость лечения, USD",
        "Медианная стоимость лечения, USD",
        "Среднее количество дней в больнице",
        "Стоимость одного выздоровевшего пациента"
    ],
    "value": [
        avg_cost_usd,
        median_cost_usd,
        avg_hospital_days,
        cost_per_recovered_patient
    ]})
business_metrics

,metric,value
0,"Средняя стоимость лечения, USD",1142.33
1,"Медианная стоимость лечения, USD",69.89
2,Среднее количество дней в больнице,9.77
3,Стоимость одного выздоровевшего пациента,364.99


Вывод:
1. Распределение стоимости сильно скошено вправо: большинство пациентов имеют относительно небольшую стоимость лечения, но есть дорогие критические случаи, которые сильно поднимают среднее.

### 4. Вероятностный анализ

In [21]:
# Поиск групп с высокой вероятностью смерти
data = ['severity',
    'disease_category',
    'bmi_group',
    'has_comorbidity',
    'alcohol_use_clean',
    'hospital_type',
    'insurance_status']
def mortality_by_group(df, group_col):
    result = (
        df.loc[df["death_flag"].notna() & df[group_col].notna(), [group_col, "death_flag"]]
        .groupby(group_col, as_index=False)
        .agg(
            cnt=("death_flag", "size"),  # число строк в группе
            p_death=("death_flag", "mean")))  # среднее по death_flag(доля смертей)
    result["p_death"] = result["p_death"].round(4)
    result["p_death_pct"] = (result["p_death"] * 100).round(2)
    result = result.sort_values("p_death", ascending=False)  # по убыванию вероятности смерти
    return result
for i in data:
    print(mortality_by_group(df, i))
    print()

   severity   cnt  p_death  p_death_pct
0  Critical  2329   0.4100        41.00
3    Severe  6305   0.0925         9.25
2  Moderate  6516   0.0259         2.59
1      Mild  4850   0.0237         2.37

   disease_category   cnt  p_death  p_death_pct
3  Non-Communicable  4970   0.2201        22.01
4       Respiratory  3112   0.1006        10.06
1        Infectious  3097   0.0659         6.59
5      Vector-Borne  3794   0.0406         4.06
0           Genetic  1232   0.0179         1.79
2     Mental Health  1265   0.0126         1.26
6        Waterborne  2530   0.0075         0.75

  bmi_group   cnt  p_death  p_death_pct
6      >=40    12   0.2500        25.00
3     30-34   456   0.1886        18.86
4     35-39   398   0.1809        18.09
5       <15   264   0.0947         9.47
1     20-24  6878   0.0899         8.99
2     25-29  6899   0.0897         8.97
0     15-19  5093   0.0783         7.83

   has_comorbidity    cnt  p_death  p_death_pct
1                1  13401   0.0966         9.

Вывод:
1. Самый сильный вероятностный сигнал - severity.
2. Вероятность смерти возрастает у Critical-пациентов, что очевидно.
3. Категории болейзней(disease_category) и побочные заболевания(comorbidity) тоже дают различия(слабее).


### 5. Проверка статистических гипотез

Гипотеза 1: смертность зависит от тяжести(severity)

In [22]:
contingency = (
    df.loc[
        df["severity"].notna() & df["death_flag"].notna(),
        ["severity", "death_flag"]]
    .groupby(["severity", "death_flag"])
    .size()
    .reset_index(name="cnt")
    .pivot(index="severity", columns="death_flag", values="cnt")
    .fillna(0))

table = contingency.to_numpy()
chi2, p_value, dof, expected = chi2_contingency(table)
print("chi2:", chi2)
print("p_value:", p_value)
print("dof:", dof)

chi2: 3461.6671023268455
p_value: 0.0
dof: 3


Гипотеза 2: стоимость лечения отличается между hospital_type

In [23]:
groups = []
for hospital in df["hospital_type"].dropna().unique():
    values = (
        df.loc[
            (df["hospital_type"] == hospital) &
            (df["treatment_cost_usd"].notna()),
            "treatment_cost_usd"]
        .to_numpy()
        .ravel())
    groups.append(values)
h_stat, p_value = kruskal(*groups)
print("H-stat:", h_stat)
print("p_value:", p_value)

H-stat: 2820.767404318524
p_value: 0.0


Гипотеза 3: recovery_days отличается между disease_category

In [24]:
recovered_df = df[df["recovery_days"].notna()]
groups = []
for category in recovered_df["disease_category"].dropna().unique():
    values = (
        recovered_df.loc[
            recovered_df["disease_category"] == category,
            "recovery_days"]
        .to_numpy()
        .ravel())
    groups.append(values)
h_stat, p_value = kruskal(*groups)
print("H-stat:", h_stat)
print("p_value:", p_value)

H-stat: 292.6113926990624
p_value: 3.881003240919606e-61


Вывод:
1. p-value показывает, что различия статистически значимы, но для практического смысла нужен effect size и сравнение конкретных групп.

### 6. Effect size
Проверка насколько связи сильные и важные для расчетов

Главные статистики:
- Cramér's V

In [25]:
# статистика из Chi-square + таблица сопряженности
def cramers_v(chi2, table):
    n = table.sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))
chi2_severity, p_value_severity, dof_severity, expected_severity = chi2_contingency(table)
cramers_v_severity = cramers_v(chi2_severity, table)
print("Cramer's V:", cramers_v_severity)  # Cила связи между двумя категориальными признаками

Cramer's V: 0.4160328774464132


Вывод:
1. Cramer_V = 0.416 - это связь средней силы между тяжестью(severity) и смертностью(death_flag).

### 7. Множественные сравнения

Поиск различающихся групп.

In [26]:
# Попарное сравнение mortality rate между severity-группами
def pairwise_proportion_tests(df, group_col, binary_col, alpha=0.05, correction="bonferroni"):
    data = df[[group_col, binary_col]].dropna().copy()
    groups = data[group_col].unique()

    results = []

    for g1, g2 in combinations(groups, 2):
        d1 = data[data[group_col] == g1]
        d2 = data[data[group_col] == g2]

        count = [d1[binary_col].sum(), d2[binary_col].sum()]
        nobs = [len(d1), len(d2)]

        stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative="two-sided")

        results.append({
            "group_1": g1,
            "group_2": g2,
            "n_1": nobs[0],  # размер сравниваемых групп
            "n_2": nobs[1],  # размер сравниваемых групп
            "rate_1": d1[binary_col].mean(),  # смертность в каждой группе
            "rate_2": d2[binary_col].mean(),  # смертность в каждой группе
            "z_stat": stat,
            "p_value": p_value  # обычный p-value
        })

    results_df = pd.DataFrame(results)

    reject, p_adj, _, _ = multipletests(
        results_df["p_value"],
        alpha=alpha,
        method=correction
    )

    results_df["p_value_adj"] = p_adj  # p-value после поправки Bonferroni
    results_df["significant"] = reject  # сохранилась ли значимость после поправки

    return results_df.sort_values("p_value_adj")

In [27]:
pairwise_severity_mortality = pairwise_proportion_tests(
    df=df,
    group_col="severity",
    binary_col="death_flag",
    correction="bonferroni"
)

pairwise_severity_mortality

,group_1,group_2,n_1,n_2,rate_1,rate_2,z_stat,p_value,p_value_adj,significant
3,Mild,Critical,4850,2329,0.023711,0.410047,-43.030431,0.000000e+00,0.000000e+00,True
5,Critical,Moderate,2329,6516,0.410047,0.025936,47.770644,0.000000e+00,0.000000e+00,True
1,Severe,Critical,6305,2329,0.092466,0.410047,-34.229674,8.752537e-257,5.251522e-256,True
2,Severe,Moderate,6305,6516,0.092466,0.025936,16.027578,8.201745e-58,4.921047e-57,True
0,Severe,Mild,6305,4850,0.092466,0.023711,14.863495,5.687858e-50,3.412715e-49,True
4,Mild,Moderate,4850,6516,0.023711,0.025936,-0.751608,4.522870e-01,1.000000e+00,False


Вывод:
1. Critical статистически отличается по смертности от всех остальных групп.
2. Severe также отличается от Mild и Moderate.
3. Mild и Moderate между собой значимо не отличаются.

### 8. Bootstrap

Устойчивость ключевой метрики.

In [28]:
# Оценка смертности в разных диапазонах
values = df["death_flag"].dropna().to_numpy()
bootstrap_rates = []
for _ in range(1000):
    sample = np.random.choice(values, size=len(values), replace=True)
    bootstrap_rates.append(sample.mean())

ci_low = np.percentile(bootstrap_rates, 2.5)
ci_high = np.percentile(bootstrap_rates, 97.5)

print("Mortality Rate:", values.mean())
print("95% Bootstrap CI:", ci_low, ci_high)

Mortality Rate: 0.0911
95% Bootstrap CI: 0.0871475 0.09505124999999999


Вывод:
1. Ообщая смертность находится в диапазоне 8.7% - 9.5%.
2. Интервал узкий, следовательно, оценка смертности(mortality rate) устойчива.

### 9. A/B тест

Разность стоимости лечения у пациентов без страховки и со страховкой.

A - пациенты без страховки

B - пациенты со страховкой

Целевая метрика: средняя стоимость лечения в долларах

Не настоящий A/B тест(сравнение уже существующих групп).

In [29]:
# Незастрахованные/застрахованные
df_ab = df.copy()
df_ab["ab_group"] = np.where(
    df_ab["insurance_status"] == "Uninsured",
    "A_Uninsured",
    "B_Insured")

In [30]:
# Проверяем размер групп
df_ab["ab_group"].value_counts()

,count
ab_group,
B_Insured,11925
A_Uninsured,8075


In [31]:
# В %
df_ab["ab_group"].value_counts(normalize=True) * 100

,proportion
ab_group,
B_Insured,59.625
A_Uninsured,40.375


In [32]:
# Основные метрики по группам
ab_summary = (
    df_ab
    .groupby("ab_group")
    .agg(
        cnt=("patient_id", "count"),  # размер группы
        avg_cost_usd=("treatment_cost_usd", "mean"),  # средняя стоимость лечения
        median_cost_usd=("treatment_cost_usd", "median"),  # медианная стоимость лечения
        std_cost_usd=("treatment_cost_usd", "std"),  # разброс стоимости
        min_cost_usd=("treatment_cost_usd", "min"),  # min диапазон стоимости
        max_cost_usd=("treatment_cost_usd", "max")  # max диапазон стоимости
    )
    .reset_index()
)

ab_summary

,ab_group,cnt,avg_cost_usd,median_cost_usd,std_cost_usd,min_cost_usd,max_cost_usd
0,A_Uninsured,8075,1994.107668,152.14,7635.206111,0.84,162722.26
1,B_Insured,11925,565.557623,38.42,2357.031259,0.13,63892.59


In [33]:
# Проверка разницы статистическим тестом
# Стоимость лечения имеет выбросы и скошенное распределение, поэтому используется Mann-Whitney U-test(данные не соответствуют условию нормального распределения), а не обычный t-test.
cost_a = df_ab.loc[
    (df_ab["ab_group"] == "A_Uninsured") &
    (df_ab["treatment_cost_usd"].notna()),
    "treatment_cost_usd"]
cost_b = df_ab.loc[
    (df_ab["ab_group"] == "B_Insured") &
    (df_ab["treatment_cost_usd"].notna()),
    "treatment_cost_usd"]
u_stat, p_value = mannwhitneyu(
    cost_a,
    cost_b,
    alternative="two-sided")
print("Mann-Whitney U:", u_stat)
print("p-value:", p_value)

Mann-Whitney U: 64554753.0
p-value: 0.0


In [34]:
# Разница средних
avg_cost_a = cost_a.mean()
avg_cost_b = cost_b.mean()
avg_cost_diff = avg_cost_b - avg_cost_a
print("Average cost A_Uninsured:", avg_cost_a)
print("Average cost B_Insured:", avg_cost_b)
print("Difference B - A:", avg_cost_diff)

Average cost A_Uninsured: 1994.1076681114553
Average cost B_Insured: 565.5576234800839
Difference B - A: -1428.5500446313713


In [35]:
# Bootstrap CI(оценки точности) для разницы средних
# Доверительный интервал для разницы средних между незастрахованными/застрахованными:
# 1. Среднее разность стоимости лечения у незастрахованных/застрахованных пациентов
# 2. В каком диапазоне может находиться эта разница
def bootstrap_mean_diff_ci(values_a, values_b, n_bootstrap=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    diffs = []
    values_a = np.array(values_a)
    values_b = np.array(values_b)
    for _ in range(n_bootstrap):
        sample_a = rng.choice(values_a, size=len(values_a), replace=True)
        sample_b = rng.choice(values_b, size=len(values_b), replace=True)
        diff = sample_b.mean() - sample_a.mean()
        diffs.append(diff)
    return {
        "mean_diff": values_b.mean() - values_a.mean(),
        "bootstrap_diff_mean": np.mean(diffs),  # средняя разница по всем bootstrap-повторениям
        "ci_low": np.percentile(diffs, 2.5),
        "ci_high": np.percentile(diffs, 97.5)}  # нижняя и верхняя границы 95% bootstrap-доверительного интервала
cost_diff_ci = bootstrap_mean_diff_ci(
    values_a=cost_a,
    values_b=cost_b,
    n_bootstrap=1000)
for key, value in cost_diff_ci.items():
    print(f"{key}: {value}")

mean_diff: -1428.5500446313713
bootstrap_diff_mean: -1424.9220450311088
ci_low: -1587.3140798048953
ci_high: -1249.5292427351676


1. Средняя стоимость лечения у группы B(Insured) ниже средней стоимости у группы A(Uninsured).
2. Оценка стабильна и не является результатом одной случайной конкретной выборки.
3. Разница средних лежит в диапазоне ci_low и ci_high.

In [36]:
pseudo_ab_result = pd.DataFrame({
    "metric": ["treatment_cost_usd"],
    "group_A": ["A_Uninsured"],
    "group_B": ["B_Insured"],
    "avg_A": [avg_cost_a],
    "avg_B": [avg_cost_b],
    "diff_B_minus_A": [avg_cost_diff],
    "test": ["Mann-Whitney U"],
    "p_value": [p_value],
    "bootstrap_ci_low": [cost_diff_ci["ci_low"]],
    "bootstrap_ci_high": [cost_diff_ci["ci_high"]]})
pseudo_ab_result

,metric,group_A,group_B,avg_A,avg_B,diff_B_minus_A,test,p_value,bootstrap_ci_low,bootstrap_ci_high
0,treatment_cost_usd,A_Uninsured,B_Insured,1994.107668,565.557623,-1428.550045,Mann-Whitney U,0.0,-1587.31408,-1249.529243


Вывод:
1. Пациенты со страховкой имеют меньшие расходы на лечение: на 1428.55 USD меньше в среднем.
2. Bootstrap CI для разницы не включает 0, а Mann-Whitney U-test показал статистически значимое различие распределений стоимости.

p.s. Это наблюдательное сравнение, а не настоящий A/B-тест, поэтому причинный эффект страховки не доказывается.

### 10. Выводы

1. Общая смертность составляет 9.11%, а полностью восстановившихся пациентов - 42.2%.
2. Сильным фактором смертности является тяжесть состояния: у Critical-пациентов mortality rate составляет 41%, а связь severity с death_flag имеет среднюю силу по Cramer = 0.416.
3. Critical-группа статистически отличается по смертности от всех остальных уровней severity, а Mild и Moderate между собой значимо не отличаются.
4. Средняя стоимость лечения существенно выше медианной(сильная правосторонняя асимметрия и наличие дорогих тяжёлых случаев).
5. В A/B тесте застрахованные пациенты имели меньшую среднюю стоимость лечения, чем незастрахованные.